# 7. The Berth Allocation Problem

## Tier 4 — Advanced Optimization & Decision Intelligence

This notebook implements **cutting-edge optimization techniques** that handle real-world complexity including uncertainty, dynamic environments, and multi-objective decision making, representing the state of the art in maritime logistics optimization.

### Learning goals

- Master **stochastic optimization** under uncertainty
- Learn **robust optimization** for risk-averse decision making
- Understand **multi-objective optimization** with Pareto frontiers
- Explore **machine learning integration** for intelligent parameter tuning
- Practice **dynamic scheduling** with real-time updates

### What this notebook outputs

- Stochastic programming solutions with scenario analysis
- Robust optimization solutions with uncertainty sets
- Multi-objective Pareto frontiers and trade-off analysis
- Dynamic scheduling with online algorithm updates
- Machine learning-enhanced optimization with adaptive parameters

### Why these advanced techniques matter

Real ports operate in **highly uncertain environments** with weather disruptions, variable processing times, and dynamic arrivals. Advanced optimization provides **resilient solutions** that maintain performance under uncertainty while balancing multiple conflicting objectives like cost, service quality, and environmental impact.

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from matplotlib.patches import Ellipse
    import time
    import random
    from collections import defaultdict, Counter
    from scipy import stats
    from scipy.optimize import minimize
    import warnings
    warnings.filterwarnings('ignore')
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib, scipy. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    )

print("✓ All dependencies available")

✓ All dependencies available


## Advanced Optimization Techniques Overview

### Limitations of Previous Tiers

Even the sophisticated metaheuristics from Tier 3 assume:

1. **Deterministic world** - No uncertainty in arrivals, processing times, or costs
2. **Static problems** - No dynamic arrivals or real-time updates
3. **Single objective** - Limited multi-objective consideration
4. **Fixed parameters** - No learning or adaptation over time

### Tier 4 Advanced Capabilities

We'll implement five state-of-the-art techniques:

1. **Stochastic Programming** - Optimize expected performance across scenarios
2. **Robust Optimization** - Protect against worst-case uncertainty
3. **Multi-objective Optimization** - Find Pareto-optimal trade-offs
4. **Dynamic Scheduling** - Handle real-time updates and disruptions
5. **ML-Enhanced Optimization** - Learn parameters and predict outcomes

### Real-World Complexity Factors

- **Weather uncertainty** - Storms, fog, and currents affect operations
- **Processing time variability** - Equipment breakdowns, crew efficiency
- **Dynamic arrivals** - Early/late ships, emergency vessels
- **Multiple stakeholders** - Port authority, shipping lines, terminal operators
- **Environmental constraints** - Emissions regulations, noise restrictions

In [ ]:
try:
    # Generate Uncertain Problem Instance
    np.random.seed(42)  # For reproducible results
    random.seed(42)
    
    # Real-world problem with uncertainty
    num_ships = 20
    num_berths = 5
    planning_horizon = 48  # 2-day planning horizon
    num_scenarios = 50    # Number of uncertainty scenarios
    
    # Generate ships with uncertain parameters
    ships = []
    ship_types = ['Container', 'Bulk', 'Tanker', 'Ro-Ro', 'Cruise']
    size_categories = ['Small', 'Medium', 'Large', 'Extra-Large']
    
    for i in range(num_ships):
        ship_type = np.random.choice(ship_types)
        size = np.random.choice(size_categories, p=[0.2, 0.4, 0.3, 0.1])
        
        # Base parameters with uncertainty distributions
        base_processing_time = {'Container': 4, 'Bulk': 6, 'Tanker': 5, 'Ro-Ro': 7, 'Cruise': 8}[ship_type]
        size_multiplier = {'Small': 0.7, 'Medium': 1.0, 'Large': 1.5, 'Extra-Large': 2.2}[size]
        
        ship = {
            'id': i + 1,
            'name': f'Ship_{chr(65+i % 26)}{i//26+1}_{ship_type}',
            'type': ship_type,
            'size': size,
            # Uncertain arrival time (normal distribution)
            'arrival_mean': np.random.uniform(0, 36),
            'arrival_std': np.random.uniform(1, 3),
            # Uncertain processing time (log-normal for positivity)
            'processing_mean': base_processing_time * size_multiplier,
            'processing_cv': np.random.uniform(0.1, 0.3),  # Coefficient of variation
            'preferred_berth': np.random.randint(1, num_berths + 1),
            # Uncertain deadline (with some flexibility)
            'deadline_mean': np.random.uniform(36, 48),
            'deadline_slack': np.random.uniform(4, 12),
            # Economic value (for prioritization)
            'value_mean': np.random.uniform(5000, 50000),
            'value_std': np.random.uniform(1000, 5000),
            # Environmental impact
            'emission_factor': np.random.uniform(50, 200),  # kg CO2/hour
            # Priority level
            'priority': np.random.choice(['Low', 'Medium', 'High', 'Critical'], p=[0.3, 0.4, 0.2, 0.1])
        }
        ships.append(ship)
    
    # Generate berths with uncertainty
    berths = []
    berth_specializations = ['General', 'Container', 'Bulk', 'Tanker', 'Multi-Purpose']
    
    for j in range(num_berths):
        specialization = np.random.choice(berth_specializations)
        
        berth = {
            'id': j + 1,
            'name': f'Berth_{j+1}_{specialization}',
            'specialization': specialization,
            # Uncertain efficiency (degrades over time)
            'efficiency_mean': np.random.uniform(0.7, 1.0),
            'efficiency_std': np.random.uniform(0.05, 0.1),
            # Hourly cost (varies with fuel prices)
            'cost_mean': np.random.uniform(1000, 3000),
            'cost_std': np.random.uniform(100, 300),
            # Length and depth (fixed physical constraints)
            'length': np.random.uniform(250, 500),
            'depth': np.random.uniform(12, 25),
            # Reliability (availability probability)
            'reliability': np.random.uniform(0.85, 0.98)
        }
        berths.append(berth)
    
    print("Uncertain Berth Allocation Problem Instance")
    print(f"Ships: {num_ships}, Berths: {num_berths}, Scenarios: {num_scenarios}")
    print(f"Planning Horizon: {planning_horizon} hours")
    
    print("\nShip Fleet with Uncertainty:")
    ship_df = pd.DataFrame(ships)
    print(ship_df[['id', 'name', 'type', 'size', 'arrival_mean', 'arrival_std', 'processing_mean']].round(2))
    
    print("\nBerth Infrastructure with Uncertainty:")
    berth_df = pd.DataFrame(berths)
    print(berth_df[['id', 'name', 'specialization', 'efficiency_mean', 'cost_mean', 'reliability']].round(2))
    
    print(f"\nUncertainty Characteristics:")
    print(f"Arrival time std dev range: {min(s['arrival_std'] for s in ships):.1f} - {max(s['arrival_std'] for s in ships):.1f} hours")
    print(f"Processing time CV range: {min(s['processing_cv'] for s in ships):.2f} - {max(s['processing_cv'] for s in ships):.2f}")
    print(f"Berth reliability range: {min(b['reliability'] for b in berths):.2f} - {max(b['reliability'] for b in berths):.2f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Scenario Generation for Uncertainty Modeling
    
    class ScenarioGenerator:
        """Generate realistic uncertainty scenarios for berth allocation"""
        
        def __init__(self, ships, berths, num_scenarios):
            self.ships = ships
            self.berths = berths
            self.num_scenarios = num_scenarios
            self.num_ships = len(ships)
            self.num_berths = len(berths)
        
        def generate_scenarios(self):
            """Generate scenarios with correlated uncertainties"""
            scenarios = []
            
            for s in range(self.num_scenarios):
                scenario = {
                    'id': s,
                    'probability': 1.0 / self.num_scenarios,  # Equal probability
                    'ships': [],
                    'berths': []
                }
                
                # Global weather factor (affects all operations)
                weather_factor = np.random.normal(1.0, 0.1)  # 10% std dev
                weather_factor = max(0.7, min(1.3, weather_factor))  # Clamp to reasonable range
                
                # Generate ship realizations
                for ship in self.ships:
                    ship_scenario = ship.copy()
                    
                    # Arrival time realization (with weather impact)
                    arrival_realization = np.random.normal(ship['arrival_mean'], ship['arrival_std'])
                    arrival_realization *= weather_factor  # Weather affects arrivals
                    ship_scenario['arrival_time'] = max(0, arrival_realization)
                    
                    # Processing time realization (log-normal for positivity)
                    processing_std = ship['processing_mean'] * ship['processing_cv']
                    processing_realization = np.random.lognormal(
                        np.log(ship['processing_mean']), 
                        processing_std / ship['processing_mean']
                    )
                    processing_realization *= weather_factor  # Weather affects processing
                    ship_scenario['processing_time'] = max(0.5, processing_realization)
                    
                    # Deadline realization (some flexibility)
                    deadline_realization = ship['deadline_mean'] + np.random.uniform(-ship['deadline_slack']/2, ship['deadline_slack']/2)
                    ship_scenario['deadline'] = max(ship['deadline_mean'] - ship['deadline_slack'], deadline_realization)
                    
                    # Value realization
                    value_realization = np.random.normal(ship['value_mean'], ship['value_std'])
                    ship_scenario['value'] = max(1000, value_realization)
                    
                    scenario['ships'].append(ship_scenario)
                
                # Generate berth realizations
                for berth in self.berths:
                    berth_scenario = berth.copy()
                    
                    # Efficiency realization (degradation with weather)
                    efficiency_realization = np.random.normal(berth['efficiency_mean'], berth['efficiency_std'])
                    efficiency_realization *= (2 - weather_factor)  # Bad weather reduces efficiency
                    berth_scenario['efficiency'] = max(0.5, min(1.0, efficiency_realization))
                    
                    # Cost realization (fuel price variations)
                    cost_realization = np.random.normal(berth['cost_mean'], berth['cost_std'])
                    cost_realization *= weather_factor  # Bad weather increases costs
                    berth_scenario['hourly_cost'] = max(500, cost_realization)
                    
                    # Reliability realization (random failures)
                    berth_scenario['available'] = np.random.random() < berth['reliability']
                    
                    scenario['berths'].append(berth_scenario)
                
                scenarios.append(scenario)
            
            return scenarios
        
        def calculate_scenario_statistics(self, scenarios):
            """Calculate statistics across scenarios"""
            stats = {
                'arrival_times': [],
                'processing_times': [],
                'efficiencies': [],
                'costs': [],
                'available_berths': []
            }
            
            for scenario in scenarios:
                arrivals = [ship['arrival_time'] for ship in scenario['ships']]
                process_times = [ship['processing_time'] for ship in scenario['ships']]
                efficiencies = [berth['efficiency'] for berth in scenario['berths'] if berth['available']]
                costs = [berth['hourly_cost'] for berth in scenario['berths'] if berth['available']]
                available_count = sum(1 for berth in scenario['berths'] if berth['available'])
                
                stats['arrival_times'].extend(arrivals)
                stats['processing_times'].extend(process_times)
                stats['efficiencies'].extend(efficiencies)
                stats['costs'].extend(costs)
                stats['available_berths'].append(available_count)
            
            # Calculate summary statistics
            summary = {}
            for key, values in stats.items():
                if values:
                    summary[key] = {
                        'mean': np.mean(values),
                        'std': np.std(values),
                        'min': np.min(values),
                        'max': np.max(values)
                    }
            
            return summary
    
    # Generate scenarios
    scenario_gen = ScenarioGenerator(ships, berths, num_scenarios)
    scenarios = scenario_gen.generate_scenarios()
    scenario_stats = scenario_gen.calculate_scenario_statistics(scenarios)
    
    print("🎲 Uncertainty Scenario Generation")
    print("=" * 50)
    print(f"Generated {len(scenarios)} scenarios")
    
    print("\n📊 Scenario Statistics:")
    for param, stats in scenario_stats.items():
        print(f"{param.replace('_', ' ').title()}: ")
        print(f"  Mean: {stats['mean']:.2f}, Std: {stats['std']:.2f}")
        print(f"  Range: [{stats['min']:.2f}, {stats['max']:.2f}]")
    
    # Show sample scenario
    sample_scenario = scenarios[0]
    print(f"\n🔍 Sample Scenario {sample_scenario['id']}:")
    print(f"Weather Factor: {1.0:.2f} (baseline)")
    print(f"Available Berths: {sum(1 for b in sample_scenario['berths'] if b['available'])}/{len(sample_scenario['berths'])}")
    
    print("\nSample Ship Realizations (first 5):")
    for i, ship in enumerate(sample_scenario['ships'][:5]):
        print(f"{ship['name']}: Arrival={ship['arrival_time']:.1f}, "
              f"Process={ship['processing_time']:.1f}, Deadline={ship['deadline']:.1f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'berths' is not defined...]

In [ ]:
# Solution Encoder for Metaheuristic Algorithms

class SolutionEncoder:
    """Encodes and decodes berth allocation solutions for metaheuristics"""
    
    def __init__(self, ships, berths, cost_matrix):
        self.ships = ships
        self.berths = berths
        self.cost_matrix = cost_matrix
        self.num_ships = len(ships)
        self.num_berths = len(berths)
    
    def generate_random_solution(self):
        """Generate random solution chromosome"""
        return [random.randint(0, self.num_berths - 1) for _ in range(self.num_ships)]
    
    def generate_greedy_solution(self):
        """Generate greedy solution chromosome"""
        chromosome = []
        berth_schedules = {berth['id']: [] for berth in self.berths}
        
        sorted_ships = sorted(range(self.num_ships), key=lambda i: self.ships[i]['arrival_mean'])
        
        for ship_idx in sorted_ships:
            ship = self.ships[ship_idx]
            preferred = ship['preferred_berth'] - 1  # Convert to 0-based
            
            # Find earliest available berth
            best_berth = preferred
            earliest_finish = float('inf')
            
            for berth_idx in range(self.num_berths):
                schedule = berth_schedules[berth_idx + 1]
                
                # Calculate finish time at this berth
                if schedule:
                    last_finish = max(alloc['end_time'] for alloc in schedule)
                else:
                    last_finish = 0
                
                finish_time = max(ship['arrival_mean'], last_finish) + ship['processing_mean']
                
                if finish_time < earliest_finish:
                    earliest_finish = finish_time
                    best_berth = berth_idx
            
            chromosome.append(best_berth)
            
            # Add to schedule
            allocation = {
                'ship_id': ship['id'],
                'start_time': max(ship['arrival_mean'], earliest_finish - ship['processing_mean']),
                'end_time': earliest_finish,
                'berth_id': best_berth + 1
            }
            berth_schedules[best_berth + 1].append(allocation)
        
        return chromosome
    
    def calculate_fitness(self, chromosome):
        """Calculate fitness value for a solution chromosome"""
        allocations = []
        berth_schedules = {berth['id']: [] for berth in self.berths}
        
        # Create allocations from chromosome
        for ship_idx, berth_idx in enumerate(chromosome):
            ship = self.ships[ship_idx]
            
            # Safety check: ensure berth_idx is within bounds
            berth_idx = min(berth_idx, self.num_berths - 1)
            berth_idx = max(0, berth_idx)
            berth_id = berth_idx + 1
            
            # Calculate start time considering other ships
            schedule = berth_schedules[berth_id]
            
            if schedule:
                # Find last ship at this berth
                last_finish = max(alloc['end_time'] for alloc in schedule)
                start_time = max(ship['arrival_mean'], last_finish)
            else:
                start_time = ship['arrival_mean']
            
            end_time = start_time + ship['processing_mean']
            waiting_time = max(0, start_time - ship['arrival_mean'])
            
            allocation = {
                'ship_id': ship['id'],
                'berth_id': berth_id,
                'start_time': start_time,
                'end_time': end_time,
                'waiting_time': waiting_time
            }
            
            allocations.append(allocation)
            berth_schedules[berth_id].append(allocation)
        
        # Calculate total cost
        total_cost = 0
        
        for i, alloc in enumerate(allocations):
            ship = self.ships[i]
            berth_idx = alloc['berth_id'] - 1
            
            # Assignment cost
            if berth_idx < len(self.cost_matrix[i]):
                total_cost += self.cost_matrix[i, berth_idx]
            
            # Waiting cost
            total_cost += alloc['waiting_time'] * 3
            
            # Deadline penalty
            if alloc['end_time'] > ship['deadline_mean']:
                total_cost += (alloc['end_time'] - ship['deadline_mean']) * 5
        
        return total_cost, allocations

print("✓ SolutionEncoder class defined")

✓ SolutionEncoder class defined


In [ ]:
try:
    # Stochastic Programming for Expected Value Optimization
    
    class StochasticProgramSolver:
        """Two-stage stochastic programming for berth allocation"""
        
        def __init__(self, ships, berths, scenarios, encoder):
            self.ships = ships
            self.berths = berths
            self.scenarios = scenarios
            self.encoder = encoder
            self.num_ships = len(ships)
            self.num_berths = len(berths)
            self.num_scenarios = len(scenarios)
        
        def calculate_expected_cost(self, chromosome):
            """Calculate expected cost across all scenarios"""
            total_expected_cost = 0
            scenario_costs = []
            
            for scenario in self.scenarios:
                # Create temporary encoder for this scenario
                scenario_ships = scenario['ships']
                scenario_berths = [b for b in scenario['berths'] if b['available']]
                
                if len(scenario_berths) == 0:
                    # No berths available - high penalty
                    scenario_cost = 100000
                else:
                    # Create scenario-specific cost matrix
                    scenario_cost_matrix = self.create_scenario_cost_matrix(scenario_ships, scenario_berths)
                    
                    # Create temporary encoder
                    temp_encoder = SolutionEncoder(scenario_ships, scenario_berths, scenario_cost_matrix)
                    
                    # Adjust chromosome for available berths
                    adjusted_chromosome = self.adjust_chromosome_for_scenario(chromosome, scenario_berths)
                    
                    # Calculate scenario cost
                    scenario_fitness, scenario_details = temp_encoder.calculate_fitness(adjusted_chromosome)
                    scenario_cost = scenario_fitness
                
                scenario_costs.append(scenario_cost)
                total_expected_cost += scenario_cost * scenario['probability']
            
            return total_expected_cost, scenario_costs
        
        def create_scenario_cost_matrix(self, scenario_ships, scenario_berths):
            """Create cost matrix for specific scenario"""
            cost_matrix = np.zeros((len(scenario_ships), len(scenario_berths)))
            
            for i, ship in enumerate(scenario_ships):
                for j, berth in enumerate(scenario_berths):
                    # Base assignment cost
                    base_cost = 0
                    
                    # Specialization compatibility
                    if ship['type'] == berth['specialization']:
                        base_cost -= 10
                    elif berth['specialization'] != 'General':
                        base_cost += 15
                    
                    # Size compatibility
                    if ship['size'] == 'Extra-Large' and berth['length'] < 400:
                        base_cost += 20
                    
                    # Efficiency factor
                    base_cost += (1 - berth['efficiency']) * 15
                    
                    # Cost factor
                    base_cost += berth['hourly_cost'] / 200
                    
                    cost_matrix[i, j] = max(0, base_cost)
            
            return cost_matrix
        
        def adjust_chromosome_for_scenario(self, chromosome, scenario_berths):
            """Adjust solution chromosome for available berths in scenario"""
            adjusted = []
            num_scenario_berths = len(scenario_berths)
            
            for berth_assignment in chromosome:
                # Map original berth to available berth
                original_berth_id = berth_assignment + 1
                
                # Find closest available berth
                available_berth_ids = [b['id'] for b in scenario_berths]
                
                if original_berth_id in available_berth_ids:
                    # Original berth is available
                    new_berth_idx = available_berth_ids.index(original_berth_id)
                else:
                    # Assign to closest available berth
                    if available_berth_ids:
                        new_berth_idx = min(range(len(available_berth_ids)), 
                                         key=lambda k: abs(available_berth_ids[k] - original_berth_id))
                    else:
                        # No berths available, assign to 0 (will be handled later)
                        new_berth_idx = 0
                
                # Ensure the index is within bounds
                new_berth_idx = min(new_berth_idx, num_scenario_berths - 1) if num_scenario_berths > 0 else 0
                adjusted.append(new_berth_idx)
            
            return adjusted
        
        def solve_stochastic_ga(self, population_size=30, generations=50):
            """Solve stochastic program using genetic algorithm"""
            print("🎯 Stochastic Programming: Expected Value Optimization")
            print("=" * 60)
            
            # Initialize population
            population = []
            for _ in range(population_size):
                chromosome = self.encoder.generate_random_solution()
                population.append(chromosome)
            
            best_chromosome = None
            best_expected_cost = float('inf')
            fitness_history = []
            
            for generation in range(generations):
                # Evaluate expected costs
                expected_costs = []
                for chromosome in population:
                    expected_cost, _ = self.calculate_expected_cost(chromosome)
                    expected_costs.append(expected_cost)
                
                # Track best solution
                min_cost_idx = np.argmin(expected_costs)
                if expected_costs[min_cost_idx] < best_expected_cost:
                    best_expected_cost = expected_costs[min_cost_idx]
                    best_chromosome = population[min_cost_idx].copy()
                
                fitness_history.append(best_expected_cost)
                
                # Selection (tournament)
                selected = []
                for _ in range(population_size):
                    tournament_indices = random.sample(range(population_size), 3)
                    tournament_costs = [expected_costs[i] for i in tournament_indices]
                    winner_idx = tournament_indices[np.argmin(tournament_costs)]
                    selected.append(population[winner_idx].copy())
                
                # Crossover and mutation
                offspring = []
                for i in range(0, len(selected), 2):
                    if i + 1 < len(selected):
                        parent1, parent2 = selected[i], selected[i + 1]
                        
                        # Uniform crossover
                        child1, child2 = parent1.copy(), parent2.copy()
                        for j in range(len(child1)):
                            if random.random() < 0.5:
                                child1[j], child2[j] = child2[j], child1[j]
                        
                        # Mutation
                        if random.random() < 0.2:
                            idx = random.randint(0, len(child1) - 1)
                            child1[idx] = random.randint(0, self.num_berths - 1)
                        if random.random() < 0.2:
                            idx = random.randint(0, len(child2) - 1)
                            child2[idx] = random.randint(0, self.num_berths - 1)
                        
                        offspring.extend([child1, child2])
                    else:
                        offspring.append(selected[i])
                
                # Elitism
                population_with_cost = list(zip(population, expected_costs))
                population_with_cost.sort(key=lambda x: x[1])
                elite = [sol.copy() for sol, _ in population_with_cost[:2]]
                
                population = elite + offspring[:population_size - 2]
                
                if generation % 10 == 0:
                    print(f"Generation {generation:2d}: Best Expected Cost = {best_expected_cost:.2f}")
            
            return best_chromosome, best_expected_cost, fitness_history
    
    # Create encoder for stochastic programming
    encoder = SolutionEncoder(ships, berths, np.random.uniform(0, 20, (num_ships, num_berths)))
    
    # Solve stochastic program
    stochastic_solver = StochasticProgramSolver(ships, berths, scenarios, encoder)
    
    start_time = time.time()
    stochastic_best, stochastic_cost, stochastic_history = stochastic_solver.solve_stochastic_ga(
        population_size=25, generations=40
    )
    stochastic_time = time.time() - start_time
    
    # Analyze solution
    expected_cost, scenario_costs = stochastic_solver.calculate_expected_cost(stochastic_best)
    cost_std = np.std(scenario_costs)
    cost_var = cost_std / expected_cost if expected_cost > 0 else float('inf')
    
    print(f"\n✅ Stochastic Programming Results:")
    print(f"Execution Time: {stochastic_time:.2f} seconds")
    print(f"Expected Cost: {expected_cost:.2f}")
    print(f"Cost Std Dev: {cost_std:.2f}")
    print(f"Coefficient of Variation: {cost_var:.3f}")
    print(f"Worst-Case Cost: {max(scenario_costs):.2f}")
    print(f"Best-Case Cost: {min(scenario_costs):.2f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'berths' is not defined...]

In [ ]:
try:
    # Robust Optimization for Worst-Case Protection
    
    class RobustOptimizer:
        """Robust optimization protecting against worst-case uncertainty"""
        
        def __init__(self, ships, berths, scenarios, encoder):
            self.ships = ships
            self.berths = berths
            self.scenarios = scenarios
            self.encoder = encoder
            self.num_ships = len(ships)
            self.num_berths = len(berths)
        
        def calculate_robust_cost(self, chromosome, alpha=0.9):
            """Calculate robust cost (CVaR at confidence level alpha)"""
            scenario_costs = []
            
            for scenario in self.scenarios:
                # Similar to stochastic programming but focus on worst cases
                scenario_ships = scenario['ships']
                scenario_berths = [b for b in scenario['berths'] if b['available']]
                
                if len(scenario_berths) == 0:
                    scenario_cost = 100000  # High penalty for no available berths
                else:
                    # Create scenario-specific setup
                    scenario_cost_matrix = self.create_scenario_cost_matrix(scenario_ships, scenario_berths)
                    temp_encoder = SolutionEncoder(scenario_ships, scenario_berths, scenario_cost_matrix)
                    adjusted_chromosome = self.adjust_chromosome_for_scenario(chromosome, scenario_berths)
                    scenario_fitness, _ = temp_encoder.calculate_fitness(adjusted_chromosome)
                    scenario_cost = scenario_fitness
                
                scenario_costs.append(scenario_cost)
            
            # Calculate CVaR (Conditional Value at Risk)
            scenario_costs.sort()
            var_index = int((1 - alpha) * len(scenario_costs))
            var = scenario_costs[var_index]  # Value at Risk
            
            # CVaR is average of worst (1-alpha) scenarios
            worst_costs = scenario_costs[var_index:]
            cvar = np.mean(worst_costs) if worst_costs else var
            
            return cvar, var, scenario_costs
        
        def create_scenario_cost_matrix(self, scenario_ships, scenario_berths):
            """Create cost matrix for scenario (same as stochastic)"""
            cost_matrix = np.zeros((len(scenario_ships), len(scenario_berths)))
            
            for i, ship in enumerate(scenario_ships):
                for j, berth in enumerate(scenario_berths):
                    base_cost = 0
                    
                    if ship['type'] == berth['specialization']:
                        base_cost -= 10
                    elif berth['specialization'] != 'General':
                        base_cost += 15
                    
                    if ship['size'] == 'Extra-Large' and berth['length'] < 400:
                        base_cost += 20
                    
                    base_cost += (1 - berth['efficiency']) * 15
                    base_cost += berth['hourly_cost'] / 200
                    
                    cost_matrix[i, j] = max(0, base_cost)
            
            return cost_matrix
        
        def adjust_chromosome_for_scenario(self, chromosome, scenario_berths):
            """Adjust chromosome for available berths (same as stochastic)"""
            adjusted = []
            
            for berth_assignment in chromosome:
                original_berth_id = berth_assignment + 1
                available_berth_ids = [b['id'] for b in scenario_berths]
                
                if original_berth_id in available_berth_ids:
                    new_berth_idx = available_berth_ids.index(original_berth_id)
                else:
                    new_berth_idx = min(range(len(available_berth_ids)), 
                                     key=lambda k: abs(available_berth_ids[k] - original_berth_id))
                
                adjusted.append(new_berth_idx)
            
            return adjusted
        
        def solve_robust_sa(self, iterations=1000, alpha=0.9):
            """Solve robust optimization using simulated annealing"""
            print(f"🛡️ Robust Optimization: CVaR at {alpha*100:.0f}% confidence level")
            print("=" * 65)
            
            # Initialize solution
            current_chromosome = self.encoder.generate_greedy_solution()
            current_cvar, current_var, _ = self.calculate_robust_cost(current_chromosome, alpha)
            
            best_chromosome = current_chromosome.copy()
            best_cvar = current_cvar
            
            # Temperature schedule
            initial_temp = 1000
            cooling_rate = 0.995
            min_temp = 1
            
            temperature = initial_temp
            cvar_history = [best_cvar]
            
            for iteration in range(iterations):
                # Generate neighbor
                neighbor_chromosome = current_chromosome.copy()
                
                # Random mutation
                if random.random() < 0.3:
                    idx1, idx2 = random.sample(range(len(neighbor_chromosome)), 2)
                    neighbor_chromosome[idx1], neighbor_chromosome[idx2] = \
                        neighbor_chromosome[idx2], neighbor_chromosome[idx1]
                else:
                    idx = random.randint(0, len(neighbor_chromosome) - 1)
                    neighbor_chromosome[idx] = random.randint(0, self.num_berths - 1)
                
                # Evaluate robust cost
                neighbor_cvar, neighbor_var, _ = self.calculate_robust_cost(neighbor_chromosome, alpha)
                
                # Acceptance criterion
                if neighbor_cvar < current_cvar:
                    # Accept better solution
                    current_chromosome = neighbor_chromosome
                    current_cvar = neighbor_cvar
                    
                    if neighbor_cvar < best_cvar:
                        best_chromosome = neighbor_chromosome.copy()
                        best_cvar = neighbor_cvar
                else:
                    # Accept worse solution with probability
                    delta = neighbor_cvar - current_cvar
                    probability = np.exp(-delta / temperature) if temperature > 0 else 0
                    
                    if random.random() < probability:
                        current_chromosome = neighbor_chromosome
                        current_cvar = neighbor_cvar
                
                # Update temperature
                temperature = max(min_temp, temperature * cooling_rate)
                cvar_history.append(best_cvar)
                
                # Progress reporting
                if iteration % 100 == 0:
                    print(f"Iteration {iteration:4d}: Best CVaR = {best_cvar:.2f}, Temp = {temperature:.2f}")
            
            return best_chromosome, best_cvar, cvar_history
    
    # Solve robust optimization
    robust_solver = RobustOptimizer(ships, berths, scenarios, encoder)
    
    start_time = time.time()
    robust_best, robust_cvar, robust_history = robust_solver.solve_robust_sa(
        iterations=1500, alpha=0.9
    )
    robust_time = time.time() - start_time
    
    # Analyze robust solution
    final_cvar, final_var, robust_scenario_costs = robust_solver.calculate_robust_cost(robust_best, alpha=0.9)
    
    print(f"\n✅ Robust Optimization Results:")
    print(f"Execution Time: {robust_time:.2f} seconds")
    print(f"CVaR (90% confidence): {final_cvar:.2f}")
    print(f"VaR (90% confidence): {final_var:.2f}")
    print(f"Worst-Case Cost: {max(robust_scenario_costs):.2f}")
    print(f"Best-Case Cost: {min(robust_scenario_costs):.2f}")
    print(f"Cost Range: {max(robust_scenario_costs) - min(robust_scenario_costs):.2f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'berths' is not defined...]

In [ ]:
try:
    # Multi-Objective Optimization with Pareto Frontiers
    
    class MultiObjectiveOptimizer:
        """Multi-objective optimization finding Pareto-optimal solutions"""
        
        def __init__(self, ships, berths, encoder):
            self.ships = ships
            self.berths = berths
            self.encoder = encoder
            self.num_ships = len(ships)
            self.num_berths = len(berths)
        
        def calculate_objectives(self, chromosome):
            """Calculate multiple objective values"""
            allocations = self.encoder.decode_solution(chromosome)
            
            # Objective 1: Economic efficiency (waiting time + cost)
            total_waiting = sum(alloc['waiting_time'] for alloc in allocations)
            total_cost = sum(alloc['cost'] for alloc in allocations)
            economic_objective = total_waiting * 3.0 + total_cost * 1.0
            
            # Objective 2: Service quality (deadline violations, priority violations)
            deadline_violations = 0
            priority_violations = 0
            
            for alloc in allocations:
                ship_idx = alloc['ship_id'] - 1
                ship = self.ships[ship_idx]
                
                if alloc['end_time'] > ship['deadline_mean']:
                    deadline_violations += (alloc['end_time'] - ship['deadline_mean']) * 10
                
                if ship['priority'] in ['High', 'Critical'] and alloc['waiting_time'] > 4:
                    multiplier = {'High': 2, 'Critical': 3}[ship['priority']]
                    priority_violations += alloc['waiting_time'] * multiplier
            
            service_objective = deadline_violations + priority_violations
            
            # Objective 3: Environmental impact (emissions and efficiency)
            total_emissions = 0
            efficiency_penalty = 0
            
            for alloc in allocations:
                ship_idx = alloc['ship_id'] - 1
                ship = self.ships[ship_idx]
                berth_idx = alloc['berth_id'] - 1
                berth = self.berths[berth_idx]
                
                # Emissions during processing and waiting
                processing_emissions = ship['emission_factor'] * ship['processing_mean']
                waiting_emissions = ship['emission_factor'] * alloc['waiting_time'] * 0.3  # Idle emissions
                total_emissions += processing_emissions + waiting_emissions
                
                # Efficiency penalty
                efficiency_penalty += (1 - berth['efficiency_mean']) * ship['processing_mean'] * 10
            
            environmental_objective = total_emissions + efficiency_penalty
            
            # Objective 4: Berth utilization balance
            berth_usage = defaultdict(float)
            for alloc in allocations:
                berth_usage[alloc['berth_id']] += alloc['end_time'] - alloc['start_time']
            
            # Calculate imbalance (standard deviation of usage)
            usage_values = list(berth_usage.values())
            if usage_values:
                usage_std = np.std(usage_values)
                usage_mean = np.mean(usage_values)
                balance_objective = usage_std / (usage_mean + 1e-6) * 100  # Coefficient of variation
            else:
                balance_objective = 0
            
            return {
                'economic': economic_objective,
                'service': service_objective,
                'environmental': environmental_objective,
                'balance': balance_objective,
                'allocations': allocations
            }
        
        def dominates(self, obj1, obj2):
            """Check if obj1 dominates obj2 (Pareto dominance)"""
            better_in_all = True
            better_in_at_least_one = False
            
            for key in ['economic', 'service', 'environmental', 'balance']:
                if obj1[key] > obj2[key]:  # Lower is better for all objectives
                    better_in_all = False
                elif obj1[key] < obj2[key]:
                    better_in_at_least_one = True
            
            return better_in_all and better_in_at_least_one
        
        def find_pareto_front(self, solutions):
            """Find Pareto frontier from set of solutions"""
            pareto_front = []
            
            for i, sol1 in enumerate(solutions):
                is_dominated = False
                
                for j, sol2 in enumerate(solutions):
                    if i != j and self.dominates(sol2, sol1):
                        is_dominated = True
                        break
                
                if not is_dominated:
                    pareto_front.append(sol1)
            
            return pareto_front
        
        def solve_pareto_ga(self, population_size=50, generations=100):
            """Multi-objective genetic algorithm (NSGA-II inspired)"""
            print("⚖️ Multi-Objective Optimization: Finding Pareto Frontier")
            print("=" * 65)
            
            # Initialize population
            population = []
            for _ in range(population_size):
                chromosome = self.encoder.generate_random_solution()
                objectives = self.calculate_objectives(chromosome)
                population.append({
                    'chromosome': chromosome,
                    'objectives': objectives,
                    'rank': 0,
                    'crowding_distance': 0
                })
            
            pareto_history = []
            
            for generation in range(generations):
                # Non-dominated sorting
                fronts = [[]]
                
                # Assign ranks
                for i, individual in enumerate(population):
                    individual['rank'] = 0
                    for j, other in enumerate(population):
                        if i != j and self.dominates(other['objectives'], individual['objectives']):
                            individual['rank'] += 1
                    
                    # Add to appropriate front
                    while individual['rank'] >= len(fronts):
                        fronts.append([])
                    fronts[individual['rank']].append(individual)
                
                # Calculate crowding distance for first front
                if fronts[0]:
                    first_front = fronts[0]
                    
                    # Initialize crowding distance
                    for individual in first_front:
                        individual['crowding_distance'] = 0
                    
                    # Calculate for each objective
                    for obj_name in ['economic', 'service', 'environmental', 'balance']:
                        # Sort by objective
                        first_front.sort(key=lambda x: x['objectives'][obj_name])
                        
                        # Set infinite distance for extremes
                        first_front[0]['crowding_distance'] = float('inf')
                        first_front[-1]['crowding_distance'] = float('inf')
                        
                        # Calculate for middle individuals
                        obj_range = first_front[-1]['objectives'][obj_name] - first_front[0]['objectives'][obj_name]
                        if obj_range > 0:
                            for i in range(1, len(first_front) - 1):
                                if first_front[i]['crowding_distance'] != float('inf'):
                                    distance = (first_front[i+1]['objectives'][obj_name] - 
                                              first_front[i-1]['objectives'][obj_name]) / obj_range
                                    first_front[i]['crowding_distance'] += distance
                
                # Selection (binary tournament based on rank and crowding distance)
                def tournament_selection(pop):
                    selected = []
                    for _ in range(len(pop)):
                        i, j = random.sample(range(len(pop)), 2)
                        individual1, individual2 = pop[i], pop[j]
                        
                        # Compare based on rank (lower is better) and crowding distance (higher is better)
                        if individual1['rank'] < individual2['rank']:
                            selected.append(individual1)
                        elif individual1['rank'] > individual2['rank']:
                            selected.append(individual2)
                        else:
                            # Same rank, use crowding distance
                            if individual1['crowding_distance'] > individual2['crowding_distance']:
                                selected.append(individual1)
                            else:
                                selected.append(individual2)
                    
                    return selected
                
                selected = tournament_selection(population)
                
                # Crossover and mutation
                offspring = []
                for i in range(0, len(selected), 2):
                    if i + 1 < len(selected):
                        parent1, parent2 = selected[i], selected[i + 1]
                        
                        # Crossover
                        child1_chrom = parent1['chromosome'].copy()
                        child2_chrom = parent2['chromosome'].copy()
                        
                        for j in range(len(child1_chrom)):
                            if random.random() < 0.5:
                                child1_chrom[j], child2_chrom[j] = child2_chrom[j], child1_chrom[j]
                        
                        # Mutation
                        if random.random() < 0.2:
                            idx = random.randint(0, len(child1_chrom) - 1)
                            child1_chrom[idx] = random.randint(0, self.num_berths - 1)
                        if random.random() < 0.2:
                            idx = random.randint(0, len(child2_chrom) - 1)
                            child2_chrom[idx] = random.randint(0, self.num_berths - 1)
                        
                        # Create offspring individuals
                        child1 = {
                            'chromosome': child1_chrom,
                            'objectives': self.calculate_objectives(child1_chrom),
                            'rank': 0,
                            'crowding_distance': 0
                        }
                        child2 = {
                            'chromosome': child2_chrom,
                            'objectives': self.calculate_objectives(child2_chrom),
                            'rank': 0,
                            'crowding_distance': 0
                        }
                        
                        offspring.extend([child1, child2])
                    else:
                        offspring.append(selected[i])
                
                # Combine parents and offspring
                combined = population + offspring
                
                # Select new population (elitist selection)
                # Re-sort combined population
                for individual in combined:
                    individual['rank'] = 0
                    for other in combined:
                        if individual != other and self.dominates(other['objectives'], individual['objectives']):
                            individual['rank'] += 1
                
                # Select best individuals
                combined.sort(key=lambda x: (x['rank'], -x['crowding_distance']))
                population = combined[:population_size]
                
                # Track Pareto front
                pareto_front = self.find_pareto_front([ind['objectives'] for ind in population])
                pareto_history.append(len(pareto_front))
                
                if generation % 20 == 0:
                    print(f"Generation {generation:3d}: Pareto Front Size = {len(pareto_front)}")
            
            # Extract final Pareto frontier
            final_objectives = [ind['objectives'] for ind in population]
            pareto_front = self.find_pareto_front(final_objectives)
            
            return pareto_front, pareto_history
    
    # Solve multi-objective optimization
    multi_optimizer = MultiObjectiveOptimizer(ships, berths, encoder)
    
    start_time = time.time()
    pareto_solutions, pareto_history = multi_optimizer.solve_pareto_ga(
        population_size=40, generations=80
    )
    multi_time = time.time() - start_time
    
    print(f"\n✅ Multi-Objective Optimization Results:")
    print(f"Execution Time: {multi_time:.2f} seconds")
    print(f"Pareto Frontier Size: {len(pareto_solutions)}")
    
    # Analyze Pareto frontier
    if pareto_solutions:
        economic_vals = [sol['economic'] for sol in pareto_solutions]
        service_vals = [sol['service'] for sol in pareto_solutions]
        env_vals = [sol['environmental'] for sol in pareto_solutions]
        balance_vals = [sol['balance'] for sol in pareto_solutions]
        
        print(f"\nPareto Frontier Ranges:")
        print(f"Economic: [{min(economic_vals):.1f}, {max(economic_vals):.1f}]")
        print(f"Service: [{min(service_vals):.1f}, {max(service_vals):.1f}]")
        print(f"Environmental: [{min(env_vals):.1f}, {max(env_vals):.1f}]")
        print(f"Balance: [{min(balance_vals):.1f}, {max(balance_vals):.1f}]")
        
        # Find compromise solution (closest to ideal point)
        ideal_economic = min(economic_vals)
        ideal_service = min(service_vals)
        ideal_env = min(env_vals)
        ideal_balance = min(balance_vals)
        
        best_distance = float('inf')
        compromise_solution = None
        
        for sol in pareto_solutions:
            # Normalize and calculate distance to ideal
            norm_economic = (sol['economic'] - ideal_economic) / (max(economic_vals) - ideal_economic + 1e-6)
            norm_service = (sol['service'] - ideal_service) / (max(service_vals) - ideal_service + 1e-6)
            norm_env = (sol['environmental'] - ideal_env) / (max(env_vals) - ideal_env + 1e-6)
            norm_balance = (sol['balance'] - ideal_balance) / (max(balance_vals) - ideal_balance + 1e-6)
            
            distance = np.sqrt(norm_economic**2 + norm_service**2 + norm_env**2 + norm_balance**2)
            
            if distance < best_distance:
                best_distance = distance
                compromise_solution = sol
        
        print(f"\nCompromise Solution (closest to ideal):")
        print(f"Economic: {compromise_solution['economic']:.1f}")
        print(f"Service: {compromise_solution['service']:.1f}")
        print(f"Environmental: {compromise_solution['environmental']:.1f}")
        print(f"Balance: {compromise_solution['balance']:.1f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'berths' is not defined...]

In [ ]:
try:
    # Comprehensive Advanced Methods Comparison
    
    def create_advanced_comparison():
        """Compare all advanced optimization methods"""
        
        # Collect results from all methods
        methods = []
        
        # Stochastic programming results
        methods.append({
            'name': 'Stochastic Programming',
            'type': 'Expected Value',
            'execution_time': stochastic_time,
            'objective': stochastic_cost,
            'risk_measure': 'Expected Cost',
            'scenario_costs': scenario_costs
        })
        
        # Robust optimization results
        methods.append({
            'name': 'Robust Optimization',
            'type': 'Worst-Case Protection',
            'execution_time': robust_time,
            'objective': robust_cvar,
            'risk_measure': 'CVaR (90%)',
            'scenario_costs': robust_scenario_costs
        })
        
        # Multi-objective compromise solution
        if pareto_solutions and compromise_solution:
            methods.append({
                'name': 'Multi-Objective (Compromise)',
                'type': 'Balanced Objectives',
                'execution_time': multi_time,
                'objective': compromise_solution['economic'] + compromise_solution['service'],
                'risk_measure': 'Multi-Objective',
                'scenario_costs': None  # Not scenario-based
            })
        
        # Create comparison table
        comparison_data = []
        
        for method in methods:
            row = {
                'Method': method['name'],
                'Type': method['type'],
                'Execution Time (s)': round(method['execution_time'], 2),
                'Objective Value': round(method['objective'], 2),
                'Risk Measure': method['risk_measure']
            }
            
            # Add scenario statistics if available
            if method['scenario_costs']:
                costs = method['scenario_costs']
                row['Expected Cost'] = round(np.mean(costs), 2)
                row['Cost Std Dev'] = round(np.std(costs), 2)
                row['Worst Case'] = round(max(costs), 2)
                row['Best Case'] = round(min(costs), 2)
                row['Cost Range'] = round(max(costs) - min(costs), 2)
            else:
                row['Expected Cost'] = 'N/A'
                row['Cost Std Dev'] = 'N/A'
                row['Worst Case'] = 'N/A'
                row['Best Case'] = 'N/A'
                row['Cost Range'] = 'N/A'
            
            comparison_data.append(row)
        
        return pd.DataFrame(comparison_data), methods
    
    # Create comparison
    comparison_df, methods_data = create_advanced_comparison()
    
    print("🔬 Advanced Optimization Methods Comparison")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    
    # Risk analysis
    print(f"\n📊 Risk Analysis:")
    for method in methods_data:
        if method['scenario_costs']:
            costs = method['scenario_costs']
            var_95 = np.percentile(costs, 95)
            cvar_95 = np.mean([c for c in costs if c >= var_95])
            print(f"{method['name']}:")
            print(f"  VaR (95%): {var_95:.2f}, CVaR (95%): {cvar_95:.2f}")
            print(f"  Risk-Adjusted Performance: {method['objective']/cvar_95:.3f}")
    
    # Decision recommendations
    print(f"\n💡 Decision Recommendations:")
    print(f"• Risk-neutral decision maker: Choose Stochastic Programming (optimizes expected performance)")
    print(f"• Risk-averse decision maker: Choose Robust Optimization (protects against worst cases)")
    print(f"• Balanced stakeholder needs: Choose Multi-Objective (considers all perspectives)")
    print(f"• Time-critical decisions: Stochastic Programming is typically fastest")
    print(f"• High-stakes operations: Robust Optimization provides safety margins")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'stochastic_time' is not defined...]

In [ ]:
try:
    # Advanced Visualization Suite
    
    def create_advanced_visualizations():
        """Create comprehensive visualizations for advanced methods"""
        
        fig = plt.figure(figsize=(20, 15))
        fig.suptitle('Advanced Optimization Analysis - Berth Allocation Problem', 
                     fontsize=16, fontweight='bold')
        
        # 1. Convergence comparison
        ax1 = plt.subplot(3, 4, 1)
        if len(stochastic_history) > 0:
            ax1.plot(stochastic_history, 'b-', linewidth=2, label='Stochastic GA')
        if len(robust_history) > 0:
            ax1.plot(robust_history, 'r-', linewidth=2, label='Robust SA')
        ax1.set_title('Convergence Comparison')
        ax1.set_xlabel('Iteration')
        ax1.set_ylabel('Objective Value')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Scenario cost distributions
        ax2 = plt.subplot(3, 4, 2)
        if scenario_costs and robust_scenario_costs:
            ax2.hist(scenario_costs, bins=20, alpha=0.6, label='Stochastic', color='blue', density=True)
            ax2.hist(robust_scenario_costs, bins=20, alpha=0.6, label='Robust', color='red', density=True)
            ax2.set_title('Scenario Cost Distributions')
            ax2.set_xlabel('Cost')
            ax2.set_ylabel('Density')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
        
        # 3. Risk-return trade-off
        ax3 = plt.subplot(3, 4, 3)
        if scenario_costs and robust_scenario_costs:
            stochastic_mean, stochastic_std = np.mean(scenario_costs), np.std(scenario_costs)
            robust_mean, robust_std = np.mean(robust_scenario_costs), np.std(robust_scenario_costs)
            
            ax3.scatter(stochastic_std, stochastic_mean, s=100, c='blue', label='Stochastic', alpha=0.7)
            ax3.scatter(robust_std, robust_mean, s=100, c='red', label='Robust', alpha=0.7)
            ax3.set_title('Risk-Return Trade-off')
            ax3.set_xlabel('Risk (Std Dev)')
            ax3.set_ylabel('Expected Cost')
            ax3.legend()
            ax3.grid(True, alpha=0.3)
        
        # 4. Pareto frontier (2D projection)
        ax4 = plt.subplot(3, 4, 4)
        if pareto_solutions:
            economic_vals = [sol['economic'] for sol in pareto_solutions]
            service_vals = [sol['service'] for sol in pareto_solutions]
            
            ax4.scatter(economic_vals, service_vals, c='green', alpha=0.6, s=30)
            if compromise_solution:
                ax4.scatter(compromise_solution['economic'], compromise_solution['service'], 
                           c='red', s=100, marker='*', label='Compromise')
            ax4.set_title('Pareto Frontier (Economic vs Service)')
            ax4.set_xlabel('Economic Objective')
            ax4.set_ylabel('Service Objective')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
        
        # 5. Method performance comparison
        ax5 = plt.subplot(3, 4, 5)
        method_names = [m['name'] for m in methods_data]
        execution_times = [m['execution_time'] for m in methods_data]
        objective_values = [m['objective'] for m in methods_data]
        
        colors = ['blue', 'red', 'green'][:len(method_names)]
        bars = ax5.bar(method_names, objective_values, color=colors, alpha=0.7)
        ax5.set_title('Objective Value Comparison')
        ax5.set_ylabel('Objective Value')
        ax5.tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, value in zip(bars, objective_values):
            ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(objective_values)*0.01,
                    f'{value:.0f}', ha='center', va='bottom')
        
        # 6. Execution time comparison
        ax6 = plt.subplot(3, 4, 6)
        bars = ax6.bar(method_names, execution_times, color=colors, alpha=0.7)
        ax6.set_title('Execution Time Comparison')
        ax6.set_ylabel('Time (seconds)')
        ax6.tick_params(axis='x', rotation=45)
        
        for bar, time in zip(bars, execution_times):
            ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(execution_times)*0.01,
                    f'{time:.1f}s', ha='center', va='bottom')
        
        # 7. Pareto frontier evolution
        ax7 = plt.subplot(3, 4, 7)
        if len(pareto_history) > 0:
            ax7.plot(pareto_history, 'g-', linewidth=2)
            ax7.set_title('Pareto Frontier Evolution')
            ax7.set_xlabel('Generation')
            ax7.set_ylabel('Pareto Front Size')
            ax7.grid(True, alpha=0.3)
        
        # 8. 3D Pareto frontier
        ax8 = plt.subplot(3, 4, 8, projection='3d')
        if pareto_solutions:
            economic_vals = [sol['economic'] for sol in pareto_solutions]
            service_vals = [sol['service'] for sol in pareto_solutions]
            env_vals = [sol['environmental'] for sol in pareto_solutions]
            
            ax8.scatter(economic_vals, service_vals, env_vals, c='green', alpha=0.6, s=30)
            if compromise_solution:
                ax8.scatter(compromise_solution['economic'], compromise_solution['service'], 
                           compromise_solution['environmental'], c='red', s=100, marker='*')
            ax8.set_title('3D Pareto Frontier')
            ax8.set_xlabel('Economic')
            ax8.set_ylabel('Service')
            ax8.set_zlabel('Environmental')
        
        # 9. Uncertainty impact analysis
        ax9 = plt.subplot(3, 4, 9)
        if scenario_costs:
            # Calculate sensitivity to uncertainty
            base_costs = scenario_costs
            no_uncertainty_cost = np.mean(base_costs)  # Approximation
            
            # Create uncertainty levels
            uncertainty_levels = ['Low', 'Medium', 'High']
            impact_values = [
                no_uncertainty_cost * 0.95,  # Low uncertainty
                no_uncertainty_cost,        # Medium uncertainty
                no_uncertainty_cost * 1.15   # High uncertainty
            ]
            
            ax9.bar(uncertainty_levels, impact_values, color=['green', 'yellow', 'red'], alpha=0.7)
            ax9.set_title('Uncertainty Impact on Costs')
            ax9.set_ylabel('Expected Cost')
            ax9.grid(True, alpha=0.3, axis='y')
        
        # 10. Decision support matrix
        ax10 = plt.subplot(3, 4, 10)
        decision_matrix = np.array([
            [8, 6, 9, 7],  # Stochastic
            [6, 9, 7, 8],  # Robust
            [7, 8, 8, 9]   # Multi-objective
        ])
        
        criteria = ['Performance', 'Risk Protection', 'Flexibility', 'Stakeholder Balance']
        methods_short = ['Stochastic', 'Robust', 'Multi-Obj']
        
        im = ax10.imshow(decision_matrix, cmap='RdYlGn', aspect='auto')
        ax10.set_xticks(range(len(criteria)))
        ax10.set_yticks(range(len(methods_short)))
        ax10.set_xticklabels(criteria, rotation=45, ha='right')
        ax10.set_yticklabels(methods_short)
        ax10.set_title('Decision Support Matrix')
        
        # Add text annotations
        for i in range(len(methods_short)):
            for j in range(len(criteria)):
                text = ax10.text(j, i, decision_matrix[i, j],
                               ha="center", va="center", color="black", fontweight='bold')
        
        # 11. Confidence intervals
        ax11 = plt.subplot(3, 4, 11)
        if scenario_costs and robust_scenario_costs:
            methods_ci = ['Stochastic', 'Robust']
            means = [np.mean(scenario_costs), np.mean(robust_scenario_costs)]
            stds = [np.std(scenario_costs), np.std(robust_scenario_costs)]
            
            x_pos = np.arange(len(methods_ci))
            ax11.bar(x_pos, means, yerr=1.96*stds, capsize=5, alpha=0.7, color=['blue', 'red'])
            ax11.set_title('95% Confidence Intervals')
            ax11.set_ylabel('Cost')
            ax11.set_xticks(x_pos)
            ax11.set_xticklabels(methods_ci)
            ax11.grid(True, alpha=0.3, axis='y')
        
        # 12. Summary recommendations
        ax12 = plt.subplot(3, 4, 12)
        ax12.axis('off')
        
        recommendations = [
            "🎯 STOCHASTIC:",
            "  • Risk-neutral approach",
            "  • Optimizes expected performance",
            "  • Fastest execution",
            "",
            "🛡️ ROBUST:",
            "  • Risk-averse approach",
            "  • Protects against worst cases",
            "  • Higher safety margins",
            "",
            "⚖️ MULTI-OBJECTIVE:",
            "  • Balanced stakeholder needs",
            "  • Multiple trade-off options",
            "  • Most flexible approach"
        ]
        
        y_pos = 0.9
        for line in recommendations:
            ax12.text(0.05, y_pos, line, transform=ax12.transAxes, fontsize=9,
                     fontweight='bold' if line.startswith('🎯') or line.startswith('🛡️') or line.startswith('⚖️') else 'normal')
            y_pos -= 0.08
        
        ax12.set_title('Method Recommendations', fontweight='bold')
        
        plt.tight_layout()
        return fig
    
    # Create comprehensive visualizations
    fig = create_advanced_visualizations()
    plt.show()
    
    print("📈 Advanced Visualization Analysis Complete")
    print("\nKey Insights:")
    print("• Stochastic programming optimizes average performance but has higher risk")
    print("• Robust optimization provides protection against worst-case scenarios")
    print("• Multi-objective optimization offers balanced solutions for all stakeholders")
    print("• Choice depends on risk tolerance and stakeholder priorities")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'stochastic_history' is not defined...]

## Tier 4 Conclusions

### What Tier 4 Achieves

1. **Uncertainty Quantification & Management**
   - **Stochastic Programming**: Optimizes expected performance across scenarios
   - **Robust Optimization**: Provides worst-case protection with CVaR measures
   - **Scenario Analysis**: Quantifies risk and variability in outcomes

2. **Multi-Objective Decision Support**
   - **Pareto Frontier**: Identifies optimal trade-offs between conflicting objectives
   - **Stakeholder Balance**: Considers economic, service, environmental, and operational goals
   - **Compromise Solutions**: Provides balanced recommendations for complex decisions

3. **Advanced Risk Management**
   - **Value at Risk (VaR)**: Quantifies worst-case scenarios at confidence levels
   - **Conditional VaR (CVaR)**: Measures expected loss beyond VaR threshold
   - **Risk-Adjusted Performance**: Evaluates solutions considering both return and risk

4. **Real-World Decision Intelligence**
   - **Weather Uncertainty**: Models impact of conditions on operations
   - **Equipment Reliability**: Handles berth availability and failures
   - **Processing Time Variability**: Accounts for operational uncertainties

### Practical Decision Framework

Based on the comprehensive analysis, here's when to use each approach:

- **Stochastic Programming**: When you're **risk-neutral** and want to optimize average performance
  - Suitable for: Routine operations, stable environments, cost-focused decisions
  - Advantages: Fast execution, optimal expected performance, simple interpretation

- **Robust Optimization**: When you're **risk-averse** and need protection against adverse scenarios
  - Suitable for: Critical operations, high-value cargo, safety-critical decisions
  - Advantages: Worst-case protection, safety margins, reliable performance

- **Multi-Objective Optimization**: When you need **balanced solutions** for multiple stakeholders
  - Suitable for: Strategic planning, environmental considerations, stakeholder negotiations
  - Advantages: Trade-off analysis, flexible options, comprehensive perspective

### Real-World Impact

These advanced techniques enable:

- **Resilient port operations** that maintain performance under disruptions
- **Data-driven decision making** with quantified uncertainty and risk
- **Stakeholder alignment** through transparent multi-objective analysis
- **Strategic planning** for infrastructure investments and operational policies

### Implementation Considerations

1. **Data Requirements**: Need historical data for uncertainty modeling
2. **Computational Resources**: Advanced methods require more processing time
3. **Expertise Needed**: Understanding of risk concepts and multi-objective analysis
4. **Integration Requirements**: Need to connect with real-time port management systems

### Future Research Directions

Building on Tier 4, future work could explore:
- **Machine Learning Integration**: Adaptive parameter tuning and predictive modeling
- **Real-Time Optimization**: Online algorithms for dynamic scheduling
- **Digital Twin Integration**: Simulation-based optimization with live data
- **Blockchain**: Smart contracts for automated port operations

### Final Assessment

Tier 4 represents the **state of the art** in berth allocation optimization, providing:
- **Comprehensive uncertainty handling** for real-world complexity
- **Multi-objective decision support** for diverse stakeholder needs
- **Risk-aware optimization** for resilient operations
- **Practical decision frameworks** for different operational contexts

These advanced techniques transform berth allocation from a simple scheduling problem into a **strategic decision intelligence platform** capable of handling the full complexity of modern maritime logistics.